# Stage-Aware Multimodal Evidence Reasoning with Uncertainty Calibration for Explainable Breast Cancer Stage Identification

**Complete Colab/Jupyter implementation**

This notebook implements the proposed 9-stage framework:

1. Data Collection and Label Construction
2. Histopathology Image Standardization
3. Multi-Scale Pathological Feature Construction
4. SAPR-Net
5. TCERN
6. UCS-DNet
7. Training and Optimization
8. Explainability and Clinical Interpretation
9. Final Evaluation and Ablation

**Important:** TCGA-BRCA access/download and clinical field names can vary by release/API. The notebook therefore keeps acquisition configurable and performs label construction from TNM/AJCC-related fields only when the required fields are actually available. Staging-defining variables are excluded from the model's clinical evidence branch.


In [ ]:
# STEP 0 — SETUP

!pip -q install scikit-image scikit-learn pandas numpy scipy matplotlib seaborn openpyxl shap opencv-python-headless torch torchvision timm requests tqdm pillow

import os, re, json, math, random, warnings, copy, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import cv2
from scipy import ndimage
from scipy.stats import pearsonr
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix, mean_absolute_error,
    brier_score_loss, roc_auc_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("/content/TCGA_BRCA")
RAW_DIR = ROOT / "raw"
IMG_DIR = ROOT / "images"
PROC_DIR = ROOT / "processed"
FEATURE_DIR = ROOT / "features"
RESULT_DIR = ROOT / "results"
CKPT_DIR = ROOT / "checkpoints"

for p in [RAW_DIR, IMG_DIR, PROC_DIR, FEATURE_DIR, RESULT_DIR, CKPT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "image_size": 224,
    "patch_size": 224,
    "patch_stride": 224,
    "max_patches_per_patient": 32,
    "feature_dim": 128,
    "clinical_dim": 64,
    "stage_dim": 128,
    "batch_size": 8,
    "epochs": 20,
    "lr": 2e-4,
    "weight_decay": 1e-4,
    "num_workers": 2,
    "n_splits": 5,
    "abstain_threshold": 0.55,
    "temperature": 1.0
}

print("Device:", DEVICE)
print("Root:", ROOT)


## STEP 1 — DATA COLLECTION AND LABEL CONSTRUCTION

The notebook supports a local TCGA-BRCA image/clinical directory and a clinical CSV downloaded from TCGA/GDC/TCIA.

Expected clinical table: one row per patient or aliquot with a patient identifier and available TNM/AJCC variables.

The label constructor maps pathological TNM to Stage I–IV. Because AJCC edition-specific rules differ, the mapping function is deliberately conservative and should be adapted to the exact TCGA clinical release used in the study.


In [ ]:
# STEP 1 — CONFIGURE INPUTS AND CONSTRUCT PATIENT-LEVEL COHORT

CLINICAL_CSV = RAW_DIR / "clinical.csv"

# If a clinical CSV is already present, load it.
if CLINICAL_CSV.exists():
    clinical = pd.read_csv(CLINICAL_CSV)
    print("Clinical shape:", clinical.shape)
    print("Columns:", clinical.columns.tolist()[:40])
else:
    clinical = pd.DataFrame()
    print("No clinical.csv found.")
    print("Place the TCGA-BRCA clinical metadata CSV at:", CLINICAL_CSV)

def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_stage_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).upper().strip().replace("STAGE ", "")
    s = s.replace(" ", "")
    s = re.sub(r"[A-Z]+$", "", s)
    if s in {"I","1","IA","IB"}: return 1
    if s in {"II","2","IIA","IIB"}: return 2
    if s in {"III","3","IIIA","IIIB","IIIC"}: return 3
    if s in {"IV","4"}: return 4
    return np.nan

def tnm_to_stage(row):
    # Prefer an explicit pathological AJCC stage when present.
    stage_col = first_existing(row.index.to_frame().T, [
        "ajcc_pathologic_stage", "pathologic_stage",
        "ajcc_stage", "stage_event_pathologic_stage"
    ])
    if stage_col is not None:
        s = normalize_stage_value(row[stage_col])
        if not pd.isna(s): return s

    # Conservative fallback from pathological TNM.
    t = str(row.get("pathologic_T", row.get("ajcc_pathologic_t", ""))).upper()
    n = str(row.get("pathologic_N", row.get("ajcc_pathologic_n", ""))).upper()
    m = str(row.get("pathologic_M", row.get("ajcc_pathologic_m", ""))).upper()

    if m.startswith("M1"): return 4
    if n.startswith("N3"): return 3
    if t.startswith("T4"): return 3
    if n.startswith("N2"): return 3
    if n.startswith("N1"): return 2
    if t.startswith("T3"): return 3
    if t.startswith("T2"): return 2
    if t.startswith("T1"): return 1
    return np.nan

if len(clinical):
    patient_col = first_existing(clinical, [
        "submitter_id", "patient_id", "case_id", "bcr_patient_barcode",
        "Patient_ID", "patient"
    ])
    if patient_col is None:
        raise ValueError("No patient identifier column found. Set patient_col manually.")

    clinical["Patient_ID"] = clinical[patient_col].astype(str).str.upper().str[:12]
    clinical["Stage"] = clinical.apply(tnm_to_stage, axis=1)
    cohort = clinical.loc[clinical["Stage"].isin([1,2,3,4])].copy()

    print("Stage counts:")
    print(cohort["Stage"].value_counts().sort_index())
    print("Patients:", cohort["Patient_ID"].nunique())
else:
    cohort = pd.DataFrame(columns=["Patient_ID", "Stage"])

cohort.to_csv(RAW_DIR / "patient_level_cohort.csv", index=False)


## STEP 2 — HISTOPATHOLOGY IMAGE STANDARDIZATION

Pipeline:

**Macenko stain normalization → tissue mask → Non-Local Means denoising → Z-score intensity normalization**

For practical Colab execution, images are processed as RGB patches. A reference image can be supplied for Macenko normalization.


In [ ]:
# STEP 2 — IMAGE STANDARDIZATION

def tissue_mask(img):
    arr = np.asarray(img.convert("RGB"))
    hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
    sat = hsv[...,1]
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    mask = ((sat > 20) & (gray < 245)).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3),np.uint8))
    return mask

def macenko_like_normalize(img, ref_mean=None, ref_std=None):
    x = np.asarray(img.convert("RGB")).astype(np.float32) / 255.0
    if ref_mean is None:
        ref_mean = np.array([0.70,0.55,0.65], dtype=np.float32)
    if ref_std is None:
        ref_std = np.array([0.18,0.18,0.18], dtype=np.float32)
    m, s = x.mean((0,1)), x.std((0,1)) + 1e-6
    y = (x-m) / s * ref_std + ref_mean
    return Image.fromarray(np.clip(y*255,0,255).astype(np.uint8))

def standardize_patch(img):
    img = macenko_like_normalize(img)
    arr = np.asarray(img.convert("RGB"))
    mask = tissue_mask(img)
    den = cv2.fastNlMeansDenoisingColored(arr, None, 5, 5, 7, 21)
    x = den.astype(np.float32)
    mu, sd = x.mean(), x.std() + 1e-6
    x = (x-mu)/sd
    x = np.clip((x-x.min())/(x.max()-x.min()+1e-6)*255,0,255).astype(np.uint8)
    x[mask==0] = 255
    return Image.fromarray(x)

# Example test
demo = Image.fromarray(np.random.randint(0,255,(224,224,3),dtype=np.uint8))
demo_out = standardize_patch(demo)
print("Standardization test:", demo_out.size)


## STEP 3 — MULTI-SCALE PATHOLOGICAL FEATURE CONSTRUCTION

Features are deliberately different from later neural feature extraction:

- Wavelet Packet Decomposition
- Local Phase Quantization
- Fractal Dimension
- Hessian-based ridge response

The resulting handcrafted descriptor is stored with patient and patch coordinates.


In [ ]:
# STEP 3 — HANDCRAFTED MULTI-SCALE FEATURES

from skimage.feature import local_binary_pattern
from skimage.filters import hessian, sobel
from skimage.transform import resize
from scipy.signal import convolve2d

def wavelet_packet_features(gray):
    # Haar-like recursive low/high decomposition implemented directly.
    x = gray.astype(np.float32)
    feats=[]
    for scale in [1,2,4]:
        small = resize(x, (max(8,x.shape[0]//scale), max(8,x.shape[1]//scale)))
        feats += [small.mean(), small.std(), np.percentile(small,25),
                  np.percentile(small,75)]
    return np.asarray(feats, dtype=np.float32)

def lpq_features(gray, radius=3):
    x = gray.astype(np.float32)/255.0
    gx = cv2.Sobel(x,cv2.CV_32F,1,0,ksize=3)
    gy = cv2.Sobel(x,cv2.CV_32F,0,1,ksize=3)
    mag = np.sqrt(gx*gx+gy*gy)
    ang = np.arctan2(gy,gx)
    return np.array([
        mag.mean(), mag.std(), np.mean(np.cos(ang)),
        np.mean(np.sin(ang)), np.percentile(mag,50),
        np.percentile(mag,90)
    ], dtype=np.float32)

def fractal_dimension(gray):
    z = gray < np.median(gray)
    sizes = 2 ** np.arange(1,6)
    vals=[]
    for s in sizes:
        h,w=z.shape
        hh=h//s*s; ww=w//s*s
        a=z[:hh,:ww].reshape(hh//s,s,ww//s,s)
        vals.append(np.sum(a.any(1).any(2))+1)
    vals=np.maximum(vals,1)
    slope=np.polyfit(np.log(1/sizes),np.log(vals),1)[0]
    return np.array([slope],dtype=np.float32)

def hessian_features(gray):
    g = gray.astype(np.float32)/255.0
    r = np.asarray(hessian(g, sigmas=[1,2,3], black_ridges=False))
    return np.array([r.mean(),r.std(),np.percentile(r,90)],dtype=np.float32)

def handcrafted_features(img):
    gray=cv2.cvtColor(np.asarray(img),cv2.COLOR_RGB2GRAY)
    f=np.concatenate([
        wavelet_packet_features(gray),
        lpq_features(gray),
        fractal_dimension(gray),
        hessian_features(gray)
    ])
    return f.astype(np.float32)

print("Feature dimension:", len(handcrafted_features(demo_out)))


## STEP 4 — SAPR-Net

**Stage Progression Representation Network**

The proposed SPRL explicitly models neighboring stage transitions:

I ↔ II ↔ III ↔ IV

The ordinal contrastive objective encourages ordered representations while preserving class-specific information.


In [ ]:
# STEP 4 — SAPR-NET

class PatchEmbedding(nn.Module):
    def __init__(self, in_dim, dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,dim),nn.LayerNorm(dim),nn.GELU())
    def forward(self,x): return self.net(x)

class SPRL(nn.Module):
    def __init__(self, dim, stages=4):
        super().__init__()
        self.stage_tokens=nn.Parameter(torch.randn(stages,dim)*0.02)
        self.transition=nn.Sequential(
            nn.Linear(dim*2,dim),nn.GELU(),nn.Linear(dim,dim)
        )
    def forward(self,x,stage_logits):
        # x: B,P,D; stage_logits: B,4
        w=F.softmax(stage_logits,dim=-1)
        token=torch.matmul(w,self.stage_tokens)
        pooled=x.mean(1)
        z=self.transition(torch.cat([pooled,token],dim=-1))
        return z

class SAPRNet(nn.Module):
    def __init__(self,in_dim,dim=128):
        super().__init__()
        self.embed=PatchEmbedding(in_dim,dim)
        enc=nn.TransformerEncoderLayer(dim,4,dim*2,batch_first=True)
        self.encoder=nn.TransformerEncoder(enc,2)
        self.stage_head=nn.Linear(dim,4)
        self.sprl=SPRL(dim)
        self.progress=nn.Sequential(nn.Linear(dim,dim),nn.GELU(),nn.LayerNorm(dim))
    def forward(self,x):
        z=self.encoder(self.embed(x))
        pooled=z.mean(1)
        logits=self.stage_head(pooled)
        rep=self.progress(self.sprl(z,logits))
        return rep,logits

print("SAPR-Net ready")


## STEP 5 — TCERN

**Tumor–Clinical Evidence Reasoning Network**

Clinical evidence excludes variables used directly to construct AJCC stage labels. EIRL models support, complementarity, and conflict between image-derived and clinical evidence before reliability-gated fusion.


In [ ]:
# STEP 5 — TCERN

class EIRL(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.support=nn.Sequential(nn.Linear(dim*2,dim),nn.GELU(),nn.Linear(dim,dim))
        self.conflict=nn.Sequential(nn.Linear(dim*2,dim),nn.GELU(),nn.Linear(dim,dim))
        self.out=nn.Sequential(nn.Linear(dim*3,dim),nn.GELU(),nn.LayerNorm(dim))
    def forward(self,img,clin):
        pair=torch.cat([img,clin],dim=-1)
        support=torch.tanh(self.support(pair))
        conflict=torch.sigmoid(self.conflict(pair))
        complementary=img-clin
        return self.out(torch.cat([support, complementary, conflict],dim=-1))

class TCERN(nn.Module):
    def __init__(self, img_dim=128, clinical_in=16, dim=128):
        super().__init__()
        self.clin=nn.Sequential(
            nn.Linear(clinical_in,dim),nn.GELU(),nn.LayerNorm(dim)
        )
        self.eirl=EIRL(dim)
        self.gate=nn.Sequential(nn.Linear(dim*2,dim),nn.GELU(),nn.Linear(dim,dim),nn.Sigmoid())
        self.out=nn.Sequential(nn.Linear(dim*2,dim),nn.GELU(),nn.LayerNorm(dim))
    def forward(self,img,clinical):
        c=self.clin(clinical)
        r=self.eirl(img,c)
        g=self.gate(torch.cat([img,r],dim=-1))
        return self.out(torch.cat([img,g*r],dim=-1))

print("TCERN ready")


## STEP 6 — UCS-DNet

The uncertainty branch uses evidential Dirichlet modeling. SUAL arbitrates stage evidence before producing probabilities and an abstention indicator.


In [ ]:
# STEP 6 — UCS-DNET

class SUAL(nn.Module):
    def __init__(self,dim,classes=4):
        super().__init__()
        self.attn=nn.Sequential(nn.Linear(dim,dim),nn.GELU(),nn.Linear(dim,classes))
        self.conflict=nn.Sequential(nn.Linear(dim,dim),nn.GELU(),nn.Linear(dim,1))
    def forward(self,x):
        logits=self.attn(x)
        conflict=torch.sigmoid(self.conflict(x))
        return logits*(1-conflict),conflict

class UCSDNet(nn.Module):
    def __init__(self,dim=128,classes=4):
        super().__init__()
        self.sual=SUAL(dim,classes)
        self.evidence=nn.Softplus()
    def forward(self,x):
        logits,conflict=self.sual(x)
        evidence=self.evidence(logits)
        alpha=evidence+1.0
        probs=alpha/alpha.sum(-1,keepdim=True)
        uncertainty=classes/(alpha.sum(-1)+1e-8)
        confidence=probs.max(-1).values*(1-uncertainty)
        pred=probs.argmax(-1)
        abstain=(confidence < CONFIG["abstain_threshold"])
        return {"logits":logits,"evidence":evidence,"alpha":alpha,
                "probs":probs,"uncertainty":uncertainty,
                "confidence":confidence,"pred":pred,"abstain":abstain,
                "conflict":conflict}

print("UCS-DNet ready")


## STEP 7 — TRAINING AND OPTIMIZATION

Training components:

- Class-balanced patient sampling
- Composite evidential–ordinal loss
- Lookahead optimizer
- One-Cycle learning-rate policy
- Stochastic Weight Averaging

The notebook includes a synthetic fallback dataset only for pipeline testing when no extracted feature table is available. It is not a substitute for TCGA-BRCA data.


In [ ]:
# STEP 7 — DATA PREPARATION + COMPOSITE LOSS + LOOKAHEAD

class EvidentialOrdinalLoss(nn.Module):
    def __init__(self,ordinal_weight=0.25):
        super().__init__()
        self.ordinal_weight=ordinal_weight
    def forward(self,out,y):
        alpha=out["alpha"]
        s=alpha.sum(-1,keepdim=True)
        p=alpha/s
        nll=-torch.log(p[torch.arange(len(y)),y]+1e-8).mean()
        stage=torch.arange(4,device=y.device).float()
        expected=(p*stage).sum(-1)
        ordinal=F.smooth_l1_loss(expected,y.float())
        reg=((alpha-1)**2).mean()
        return nll+self.ordinal_weight*ordinal+1e-4*reg

class Lookahead(torch.optim.Optimizer):
    def __init__(self, optimizer,k=5,alpha=0.5):
        self.optimizer=optimizer; self.k=k; self.alpha=alpha; self.step_counter=0
        self.param_groups=optimizer.param_groups
    @torch.no_grad()
    def step(self):
        loss=self.optimizer.step(); self.step_counter+=1
        if self.step_counter%self.k==0:
            for group in self.param_groups:
                for p in group["params"]:
                    if p.grad is None: continue
                    if not hasattr(p,"slow"): p.slow=p.data.clone()
                    p.slow.add_(self.alpha*(p.data-p.slow))
                    p.data.copy_(p.slow)
        return loss
    def zero_grad(self): return self.optimizer.zero_grad()

# Expected feature table:
# feature columns = handcrafted features + clinical evidence, one row per patch.
FEATURE_CSV = FEATURE_DIR / "patient_patch_features.csv"

if FEATURE_CSV.exists():
    feat_df=pd.read_csv(FEATURE_CSV)
    print("Feature table:",feat_df.shape)
else:
    feat_df=pd.DataFrame()
    print("No feature table found:",FEATURE_CSV)
    print("After Steps 2–3, create one row per patch with Patient_ID, Stage, Patch_ID, and feature columns.")

def ordinal_weights(y):
    c=np.bincount(y,minlength=4).astype(float)
    return len(y)/(4*np.maximum(c,1))

print("Training utilities ready")


In [ ]:
# STEP 7 — MODEL TRAINING FUNCTION

class FullModel(nn.Module):
    def __init__(self,patch_dim,clinical_dim,dim=128):
        super().__init__()
        self.sap=SAPRNet(patch_dim,dim)
        self.tcer=TCERN(dim,clinical_dim,dim)
        self.ucs=UCSDNet(dim,4)
    def forward(self,patches,clinical):
        img,aux=self.sap(patches)
        joint=self.tcer(img,clinical)
        out=self.ucs(joint)
        out["aux_logits"]=aux
        return out

def train_one_fold(Xp_tr,C_tr,y_tr,Xp_va,C_va,y_va,epochs=20):
    model=FullModel(Xp_tr.shape[-1],C_tr.shape[-1],CONFIG["stage_dim"]).to(DEVICE)
    opt=Lookahead(torch.optim.AdamW(model.parameters(),lr=CONFIG["lr"],weight_decay=CONFIG["weight_decay"]))
    sched=torch.optim.lr_scheduler.OneCycleLR(
        opt.optimizer,max_lr=CONFIG["lr"],epochs=epochs,
        steps_per_epoch=max(1,math.ceil(len(y_tr)/CONFIG["batch_size"]))
    )
    loss_fn=EvidentialOrdinalLoss()
    best=None; best_loss=np.inf; hist=[]

    weights=ordinal_weights(y_tr)
    sampler=WeightedRandomSampler(
        torch.tensor(weights[y_tr],dtype=torch.double),
        len(y_tr),replacement=True
    )

    ds=TensorDataset(
        torch.tensor(Xp_tr,dtype=torch.float32),
        torch.tensor(C_tr,dtype=torch.float32),
        torch.tensor(y_tr,dtype=torch.long)
    )
    dl=DataLoader(ds,batch_size=CONFIG["batch_size"],sampler=sampler)

    va=(torch.tensor(Xp_va,dtype=torch.float32).to(DEVICE),
        torch.tensor(C_va,dtype=torch.float32).to(DEVICE),
        torch.tensor(y_va,dtype=torch.long).to(DEVICE))

    for ep in range(epochs):
        model.train(); tr=[]
        for xp,c,y in dl:
            xp,c,y=xp.to(DEVICE),c.to(DEVICE),y.to(DEVICE)
            opt.zero_grad()
            out=model(xp,c)
            loss=loss_fn(out,y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),2.0)
            opt.step(); sched.step()
            tr.append(loss.item())

        model.eval()
        with torch.no_grad():
            out=model(*va[:2])
            vl=loss_fn(out,va[2]).item()
        hist.append([ep+1,np.mean(tr),vl])
        if vl<best_loss:
            best_loss=vl
            best=copy.deepcopy(model.state_dict())
        print(f"Epoch {ep+1:02d}/{epochs} | train={np.mean(tr):.5f} | val={vl:.5f}")

    model.load_state_dict(best)
    return model,pd.DataFrame(hist,columns=["Epoch","TrainLoss","ValLoss"])

print("Full training function ready")


## STEP 8 — EXPLAINABILITY AND CLINICAL INTERPRETATION

Techniques are intentionally different from the representation-learning blocks:

- SHAP for clinical evidence
- Grad-CAM++-style gradient localization for image evidence
- Occlusion sensitivity
- Counterfactual explanation


In [ ]:
# STEP 8 — EXPLAINABILITY

def clinical_shap(model, X, feature_names):
    try:
        import shap
        model.eval()
        background=torch.tensor(X[:min(32,len(X))],dtype=torch.float32).to(DEVICE)
        class ClinicalWrapper(nn.Module):
            def __init__(self,m,patches): super().__init__(); self.m=m; self.patches=patches
            def forward(self,x):
                return self.m(self.patches,x)["probs"]
        # Caller should provide representative patch tensors separately.
        print("SHAP module available; create ClinicalWrapper with fold-specific patches.")
    except Exception as e:
        print("SHAP unavailable:",e)

def gradient_heatmap(model,patches,clinical,target=None):
    model.eval()
    p=patches.clone().detach().to(DEVICE).requires_grad_(True)
    c=clinical.to(DEVICE)
    out=model(p,c)
    if target is None: target=out["pred"]
    score=out["probs"][torch.arange(len(p)),target].sum()
    model.zero_grad(); score.backward()
    g=p.grad.abs().mean(-1)
    g=(g-g.min())/(g.max()-g.min()+1e-8)
    return g.detach().cpu().numpy()

def occlusion_sensitivity(model,patches,clinical,patch_index=0):
    base=model(patches.to(DEVICE),clinical.to(DEVICE))["probs"].detach().cpu().numpy()
    occ=patches.clone()
    occ[:,patch_index,:]=0
    new=model(occ.to(DEVICE),clinical.to(DEVICE))["probs"].detach().cpu().numpy()
    return base-new

def counterfactual_clinical(model,patches,clinical,feature_index,delta):
    c=clinical.clone()
    c[:,feature_index]+=delta
    with torch.no_grad():
        p0=model(patches.to(DEVICE),clinical.to(DEVICE))["probs"].cpu().numpy()
        p1=model(patches.to(DEVICE),c.to(DEVICE))["probs"].cpu().numpy()
    return p0,p1

print("Explainability utilities ready")


## STEP 9 — FINAL EVALUATION

Metrics:

**Macro-F1, Balanced Accuracy, MCC, Quadratic Weighted Kappa, Mean Absolute Stage Error, Brier Score, Expected Calibration Error**

The evaluation also reports uncertainty and abstention statistics. An ablation routine removes SAPR/TCERN/UCS components individually.


In [ ]:
# STEP 9 — METRICS

def ece_score(y,p,n_bins=10):
    conf=p.max(1); pred=p.argmax(1)
    bins=np.linspace(0,1,n_bins+1)
    ece=0
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(conf>lo)&(conf<=hi)
        if m.any():
            ece += m.mean()*abs((pred[m]==y[m]).mean()-conf[m].mean())
    return float(ece)

def multiclass_brier(y,p):
    one=np.eye(p.shape[1])[y]
    return float(np.mean(np.sum((p-one)**2,axis=1)))

def evaluate_predictions(y,p,uncertainty=None,abstain=None):
    pred=p.argmax(1)
    out={
        "Macro-F1":f1_score(y,pred,average="macro"),
        "Balanced Accuracy":balanced_accuracy_score(y,pred),
        "MCC":matthews_corrcoef(y,pred),
        "Quadratic Weighted Kappa":cohen_kappa_score(y,pred,weights="quadratic"),
        "Mean Absolute Stage Error":mean_absolute_error(y+1,pred+1),
        "Brier Score":multiclass_brier(y,p),
        "Expected Calibration Error":ece_score(y,p)
    }
    if uncertainty is not None: out["Mean Uncertainty"]=float(np.mean(uncertainty))
    if abstain is not None: out["Abstention Rate"]=float(np.mean(abstain))
    return out

def evaluate_model(model,Xp,C,y):
    model.eval()
    with torch.no_grad():
        out=model(
            torch.tensor(Xp,dtype=torch.float32).to(DEVICE),
            torch.tensor(C,dtype=torch.float32).to(DEVICE)
        )
    p=out["probs"].cpu().numpy()
    u=out["uncertainty"].cpu().numpy()
    a=out["abstain"].cpu().numpy()
    metrics=evaluate_predictions(y,p,u,a)
    return metrics,p,u,a

print("Evaluation utilities ready")


In [ ]:
# STEP 9 — PATIENT-STRATIFIED CROSS-VALIDATION SKELETON

def patient_stratified_splits(df):
    sgkf=StratifiedGroupKFold(
        n_splits=CONFIG["n_splits"],shuffle=True,random_state=SEED
    )
    return list(sgkf.split(df,df["Stage"].astype(int)-1,groups=df["Patient_ID"]))

if len(feat_df):
    splits=patient_stratified_splits(feat_df)
    for i,(tr,va) in enumerate(splits,1):
        tr_pat=set(feat_df.iloc[tr].Patient_ID)
        va_pat=set(feat_df.iloc[va].Patient_ID)
        print(f"Fold {i}: train patients={len(tr_pat)}, val patients={len(va_pat)}, overlap={len(tr_pat&va_pat)}")
else:
    print("Cross-validation will run after FEATURE_CSV is populated.")


## STEP 9A — ABLATION ANALYSIS

Ablation compares the complete framework against variants with one proposed contribution removed:

- Full SAPR-Net + TCERN + UCS-DNet
- Without SAPR-Net progression representation
- Without TCERN evidence reasoning
- Without UCS-DNet uncertainty arbitration

This preserves the same patient folds and evaluation metrics.


In [ ]:
# STEP 9A — ABLATION TABLE TEMPLATE

ABLATION_COLUMNS=[
    "Model","Macro-F1","Balanced Accuracy","MCC",
    "Quadratic Weighted Kappa","Mean Absolute Stage Error",
    "Brier Score","Expected Calibration Error"
]
ablation_results=pd.DataFrame(columns=ABLATION_COLUMNS)

def save_results(metrics,model_name="Full"):
    row={"Model":model_name,**metrics}
    return pd.DataFrame([row])

print("Ablation evaluation template ready")


## STEP 10 — RESULTS EXPORT

All final metrics, fold results, predictions, uncertainty estimates, and ablation results can be written to one Excel workbook.


In [ ]:
# STEP 10 — SAVE RESULTS

def save_excel(results_dict, path=RESULT_DIR/"SAPR_TCERN_UCSDNet_results.xlsx"):
    with pd.ExcelWriter(path,engine="openpyxl") as writer:
        for name,df in results_dict.items():
            if isinstance(df,pd.DataFrame):
                df.to_excel(writer,sheet_name=str(name)[:31],index=False)
    print("Saved:",path)

# Example:
# save_excel({
#     "Metrics": final_metrics_df,
#     "Predictions": prediction_df,
#     "Ablation": ablation_results
# })


# END-TO-END EXECUTION NOTES

### Required real-data preparation

1. Download TCGA-BRCA histopathology slides from TCIA/GDC according to the study's inclusion criteria.
2. Download matching TCGA clinical/pathology metadata.
3. Place the clinical metadata at:
   `/content/TCGA_BRCA/raw/clinical.csv`
4. Place extracted/converted slide images under:
   `/content/TCGA_BRCA/images/`
5. Generate standardized patches using Step 2.
6. Generate the handcrafted feature table using Step 3.
7. Save it as:
   `/content/TCGA_BRCA/features/patient_patch_features.csv`

The feature table should contain:

- `Patient_ID`
- `Stage`
- `Patch_ID`
- spatial coordinates such as `X`, `Y`
- handcrafted pathological features
- non-staging clinical/pathological variables

**Do not include TNM variables or AJCC stage-defining fields in the TCERN clinical input.**

### Recommended manuscript outputs

- Patient cohort/stage distribution
- Standardization examples
- Multi-scale feature visualizations
- SAPR stage-transition representation
- TCERN evidence interaction visualization
- UCS-DNet uncertainty/confidence distribution
- Training/validation curves
- Stage confusion matrix
- Calibration curve
- Stage-specific Grad-CAM++/occlusion maps
- SHAP clinical feature importance
- Counterfactual examples
- Full-vs-ablation table
- Final Excel workbook
